# IT25102025 — Logistic Regression

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25102025 — Logistic Regression

**Model suitability**
- The task is binary classification: will the client default next month (1) or not (0)? Logistic Regression is the standard starting model for this kind of problem.
- It predicts a probability of default, which suits credit risk work.
- It is easy to interpret: each feature's coefficient shows whether it raises or lowers default risk.
- It needs scaled features, which Stage 5 already provides.

**Implementation**
- Library: scikit-learn (`LogisticRegression`, `GridSearchCV`, `StratifiedKFold`).
- Solver: `liblinear`, which supports both L1 and L2 penalties.
- The penalty type is set with `l1_ratio` (0 = L2, 1 = L1). The older `penalty` setting is deprecated since scikit-learn 1.8.

**Parameter tuning**
- Method: `GridSearchCV`, which tries every combination below (5 × 2 × 2 = **20 models**):
  - `C` (regularization strength): 0.01, 0.1, 1, 10, 100
  - `l1_ratio` (penalty type): `0` = L2 (shrinks all coefficients) or `1` = L1 (can shrink weak features to exactly zero)
  - `class_weight`: `None` or `balanced` (only ~22% of clients default, so `balanced` makes the model pay more attention to defaulters)

**Validation method**
- 5-fold **stratified** cross-validation on the training set only. Each fold keeps the same 22% default ratio.
- The best model is then checked once on the held-out 20% test set.

**Evaluation metrics**
- **F1** (used to choose the best model): balances precision and recall. Accuracy is misleading here because predicting "no default" for everyone would already be ~78% accurate.
- **Recall**: how many real defaulters we catch. Missing a defaulter is costly for a bank.
- **Precision**: when we predict default, how often we are right.
- **ROC-AUC**: how well the model ranks risky clients above safe ones, regardless of threshold.


In [ ]:
# IT25102025 — Logistic Regression with grid search
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# 1. Settings to try: 5 C values x 2 penalty types x 2 class weights = 20 models
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'l1_ratio': [0, 1],   # 0 = L2 penalty, 1 = L1 penalty
    'class_weight': [None, 'balanced'],
}

# 2. 5-fold stratified cross-validation on the training data only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logistic_search = GridSearchCV(
    LogisticRegression(solver='liblinear', max_iter=1000),
    param_grid=param_grid,
    scoring='f1',
    cv=cv
)
logistic_search.fit(X_train, y_train)
my_logistic_model = logistic_search.best_estimator_
print(f'Best settings: {logistic_search.best_params_}')
print(f'Best cross-validation F1: {logistic_search.best_score_:.3f}')

# 3. Compare all 20 models tried
results = pd.DataFrame(logistic_search.cv_results_)
results = results[['param_C', 'param_l1_ratio', 'param_class_weight', 'mean_test_score']]
results['param_l1_ratio'] = results['param_l1_ratio'].map({0: 'L2', 1: 'L1'})
results['param_class_weight'] = results['param_class_weight'].fillna('None')
results = results.rename(columns={'param_l1_ratio': 'penalty', 'mean_test_score': 'CV F1'}).sort_values('CV F1', ascending=False)
display(results.reset_index(drop=True))

# 4. Final check on the test set
y_pred_lr = my_logistic_model.predict(X_test)
print(classification_report(y_test, y_pred_lr, target_names=['No Default', 'Default']))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=['No Default', 'Default'])
plt.title('IT25102025 Logistic Regression: Confusion Matrix')
plt.show()

# 5. Which features raise (+) or lower (-) default risk
coefs = pd.Series(my_logistic_model.coef_[0], index=X_train.columns).sort_values()
print(coefs.round(3))

record_model('IT25102025', 'Logistic Regression', my_logistic_model)


**IT25102025 — Results and conclusion**

- **Comparison of the 20 models:** all 10 models with `class_weight='balanced'` scored about **0.52 CV F1**. All 10 models without it scored only about **0.40**. Changing `C` or the penalty type (L1/L2) made almost no difference (less than 0.01).
- **Best model:** `C=0.01`, L1 penalty (`l1_ratio=1`), `class_weight='balanced'` (CV F1 ≈ 0.524).
- **Test set result:** F1 ≈ 0.52, Recall ≈ 0.61, Precision ≈ 0.45, ROC-AUC ≈ 0.75.
- **Before vs after:** the first version (3 C values, no class weighting) caught only **27%** of defaulters (F1 0.38). The tuned version catches about **61%** (F1 0.52).
- **Conclusion:** handling class imbalance mattered far more than the regularization settings. Accuracy fell slightly (0.80 → 0.75) because the model now flags more clients as risky, which is the right trade-off when missing a defaulter is costly.
- **Most important features:** `delay_count`, `PAY_0` and `max_delay` raise default risk. `avg_pay_amt` and `LIMIT_BAL` lower it.

**Limitations and possible improvements**
- Logistic Regression only draws a straight-line boundary, so it cannot capture complex patterns. Tree models (Decision Tree, Random Forest) can, and reach slightly higher F1 (about 0.53).
- The `PAY_x` features are strongly related to each other, so some individual coefficients are hard to interpret.
- The decision threshold is fixed at 0.5. Tuning it could trade precision for recall more precisely.
- Preprocessing (scaling, feature selection) was fitted before the train/test split, so test scores may be slightly optimistic.
